# Shared memory: adjacent differences

For `b[i] = a[i] - a[i-1]`, each output uses two neighboring inputs. The direct kernel reads them from global memory. The shared-memory kernel loads a block tile and its left halo, synchronizes, then computes the same outputs.

## Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → GPU**. This cell displays the selected device and creates a folder for the CUDA source files.

In [1]:
!nvidia-smi
!nvcc --version
!mkdir -p code


Sat Nov 14 06:53:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## CUDA error and timing support

The error-checking macro is independent of the timer. `GetCounter()` returns elapsed milliseconds and waits for the recorded work to finish.

In [2]:
%%writefile code/cuda_check.cuh
#ifndef CUDA_CHECK_CUH
#define CUDA_CHECK_CUH
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#define CUDA_CHECK(call) do { \
    cudaError_t status = (call); \
    if (status != cudaSuccess) { \
        std::fprintf(stderr, "%s:%d: %s\n", __FILE__, __LINE__, \
                     cudaGetErrorString(status)); \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)
#endif


Writing code/cuda_check.cuh


In [3]:
%%writefile code/TimingGPU.cuh
#ifndef TIMING_GPU_CUH
#define TIMING_GPU_CUH
#include <cuda_runtime.h>
class TimingGPU {
    cudaEvent_t start_;
    cudaEvent_t stop_;
    bool started_;
public:
    TimingGPU();
    ~TimingGPU();
    TimingGPU(const TimingGPU&) = delete;
    TimingGPU& operator=(const TimingGPU&) = delete;
    void StartCounter();
    float GetCounter(); // milliseconds
};
#endif


Writing code/TimingGPU.cuh


In [4]:
%%writefile code/TimingGPU.cu
#include "TimingGPU.cuh"
#include "cuda_check.cuh"
#include <cstdio>
#include <cstdlib>

TimingGPU::TimingGPU() : started_(false) {
    CUDA_CHECK(cudaEventCreate(&start_));
    CUDA_CHECK(cudaEventCreate(&stop_));
}

TimingGPU::~TimingGPU() {
    CUDA_CHECK(cudaEventDestroy(start_));
    CUDA_CHECK(cudaEventDestroy(stop_));
}

void TimingGPU::StartCounter() {
    CUDA_CHECK(cudaEventRecord(start_));
    started_ = true;
}

float TimingGPU::GetCounter() {
    if (!started_) {
        std::fprintf(stderr, "Call StartCounter before GetCounter.\n");
        std::exit(EXIT_FAILURE);
    }
    CUDA_CHECK(cudaEventRecord(stop_));
    CUDA_CHECK(cudaEventSynchronize(stop_));
    float milliseconds = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&milliseconds, start_, stop_));
    started_ = false;
    return milliseconds;
}


Writing code/TimingGPU.cu


## Kernels and host program

The direct and shared-memory kernels compute the same adjacent differences. The host also builds a CPU reference array and compares every GPU value with it. In the shared kernel, every thread loads its assigned value and reaches `__syncthreads()`, including threads in a partial last block.

In [5]:
%%writefile code/chapter02_adjacent_shared.cu
#include "cuda_check.cuh"
#include "TimingGPU.cuh"
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <cstring>

__global__ void adjacent_direct(const float *a, float *b, int n) {
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) b[i] = (i == 0) ? 0.0f : a[i] - a[i - 1];
}

__global__ void adjacent_shared(const float *a, float *b, int n) {
    extern __shared__ float tile[];
    const int t = threadIdx.x;
    const int i = blockIdx.x * blockDim.x + t;
    tile[t + 1] = (i < n) ? a[i] : 0.0f;
    if (t == 0) {
        tile[0] = (i > 0 && i < n) ? a[i - 1] : 0.0f;
    }
    __syncthreads();
    if (i < n) b[i] = (i == 0) ? 0.0f : tile[t + 1] - tile[t];
}

void launch(int shared, const float *d_a, float *d_b, int n, int threads) {
    const int blocks = (n + threads - 1) / threads;
    if (shared) {
        const size_t bytes = size_t(threads + 1) * sizeof(float);
        adjacent_shared<<<blocks, threads, bytes>>>(d_a, d_b, n);
    } else {
        adjacent_direct<<<blocks, threads>>>(d_a, d_b, n);
    }
    CUDA_CHECK(cudaGetLastError());
}

void run(int n, int threads, bool show, int repeats) {
    const size_t bytes = size_t(n) * sizeof(float);
    float *h_a = (float*)std::malloc(bytes);
    float *h_cpu = (float*)std::malloc(bytes);
    float *h_direct = (float*)std::malloc(bytes);
    float *h_shared = (float*)std::malloc(bytes);
    float *d_a = nullptr, *d_direct = nullptr, *d_shared = nullptr;
    if (!h_a || !h_cpu || !h_direct || !h_shared) {
        std::fprintf(stderr, "Host allocation failed.\n");
        std::exit(EXIT_FAILURE);
    }
    for (int i = 0; i < n; ++i) h_a[i] = float((i % 101) - 50) / 10.0f;
    h_cpu[0] = 0.0f;
    for (int i = 1; i < n; ++i) h_cpu[i] = h_a[i] - h_a[i - 1];
    CUDA_CHECK(cudaMalloc((void**)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_direct, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_shared, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));

    launch(0, d_a, d_direct, n, threads);
    launch(1, d_a, d_shared, n, threads);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_direct, d_direct, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_shared, d_shared, bytes, cudaMemcpyDeviceToHost));
    for (int i = 0; i < n; ++i) {
        if (std::fabs(h_direct[i] - h_cpu[i]) > 1.0e-6f) {
            std::fprintf(stderr, "CPU/direct-GPU mismatch at i=%d.\n", i);
            std::exit(EXIT_FAILURE);
        }
        if (std::fabs(h_shared[i] - h_cpu[i]) > 1.0e-6f) {
            std::fprintf(stderr, "CPU/shared-GPU mismatch at i=%d.\n", i);
            std::exit(EXIT_FAILURE);
        }
    }
    std::printf("CPU vs direct GPU: PASS (%d values)\n", n);
    std::printf("CPU vs shared GPU: PASS (%d values)\n", n);
    if (show) {
        std::printf(" i       a[i]        CPU    direct    shared\n");
        for (int i = 0; i < n; ++i) {
            std::printf("%2d %10.3f %9.3f %9.3f %9.3f\n",
                        i, h_a[i], h_cpu[i], h_direct[i], h_shared[i]);
        }
    } else {
        TimingGPU timer;
        float ms[2];
        for (int method = 0; method < 2; ++method) {
            for (int k = 0; k < 5; ++k)
                launch(method, d_a, method ? d_shared : d_direct, n, threads);
            CUDA_CHECK(cudaDeviceSynchronize());
            timer.StartCounter();
            for (int k = 0; k < repeats; ++k)
                launch(method, d_a, method ? d_shared : d_direct, n, threads);
            ms[method] = timer.GetCounter() / repeats;
        }
        std::printf("direct: %.6f ms/launch, shared: %.6f ms/launch\n",
                    ms[0], ms[1]);
        std::printf("T_direct / T_shared = %.3f\n", ms[0] / ms[1]);
    }
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_direct));
    CUDA_CHECK(cudaFree(d_shared));
    std::free(h_a); std::free(h_cpu); std::free(h_direct); std::free(h_shared);
}

int main(int argc, char **argv) {
    if (argc != 2 || (std::strcmp(argv[1], "--demo") != 0 &&
                      std::strcmp(argv[1], "--time") != 0)) {
        std::fprintf(stderr, "Usage: %s --demo | --time\n", argv[0]);
        return EXIT_FAILURE;
    }
    CUDA_CHECK(cudaSetDevice(0));
    const bool show = std::strcmp(argv[1], "--demo") == 0;
    run(show ? 10 : 1048576, show ? 4 : 256, show, 100);
    return EXIT_SUCCESS;
}


Writing code/chapter02_adjacent_shared.cu


## Compile

In [6]:
%%bash
set -eu
nvcc -O2 -std=c++14 -Icode code/chapter02_adjacent_shared.cu code/TimingGPU.cu -o adjacent_demo


## Print and validate a small case

This uses N=10 and a block size of 4, so the last block is partial. The program prints the CPU reference and both GPU outputs, then checks every GPU value against the CPU array.

In [7]:
import subprocess

def run_and_show(command):
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='')
    result.check_returncode()

run_and_show(['./adjacent_demo', '--demo'])


CPU vs direct GPU: PASS (10 values)
CPU vs shared GPU: PASS (10 values)
 i       a[i]        CPU    direct    shared
 0     -5.000     0.000     0.000     0.000
 1     -4.900     0.100     0.100     0.100
 2     -4.800     0.100     0.100     0.100
 3     -4.700     0.100     0.100     0.100
 4     -4.600     0.100     0.100     0.100
 5     -4.500     0.100     0.100     0.100
 6     -4.400     0.100     0.100     0.100
 7     -4.300     0.100     0.100     0.100
 8     -4.200     0.100     0.100     0.100
 9     -4.100     0.100     0.100     0.100


## Compare execution times

The larger case uses one million elements and a block size of 256. It first prints CPU-versus-GPU validation results, then reports the average time per launch for both kernels and their ratio. The timer excludes allocation, transfers, validation, and printing.

In [8]:
run_and_show(['./adjacent_demo', '--time'])

CPU vs direct GPU: PASS (1048576 values)
CPU vs shared GPU: PASS (1048576 values)
direct: 0.036078 ms/launch, shared: 0.038436 ms/launch
T_direct / T_shared = 0.939
